## Jekyll Homework
Ananya Kavatekar

website url: https://ananyakavatekar.github.io/

In [2]:
import pandas as pd 
import altair as alt

In [3]:
data = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/licenses_fall2022.csv"
df = pd.read_csv(data)
df.shape

(10000, 31)

## EDA

In [4]:
df.head()

,_id,License Type,Description,License Number,License Status,Business,Title,First Name,Middle,Last Name,...,Specialty/Qualifier,Controlled Substance Schedule,Delegated Controlled Substance Schedule,Ever Disciplined,LastModifiedDate,Case Number,Action,Discipline Start Date,Discipline End Date,Discipline Reason
0,1189509,DETECTIVE BOARD,PERMANENT EMPLOYEE REGISTRATION,129446286,NOT RENEWED,N,NaN,EILEEN,NaN,SANTACRUZ,...,NaN,NaN,NaN,N,03/18/2022,NaN,NaN,NaN,NaN,NaN
1,801037,DETECTIVE BOARD,FIREARM CONTROL CARD,229030294.0,NOT RENEWED,N,NaN,DAGMAR,J,NORDLUND,...,NaN,NaN,NaN,N,08/16/2006,NaN,NaN,NaN,NaN,NaN
2,365129,COSMO,LICENSED COSMETOLOGIST,11053076.0,NOT RENEWED,N,NaN,RADOJE,NaN,ZELENOVIC,...,NaN,NaN,NaN,N,05/26/2006,NaN,NaN,NaN,NaN,NaN
3,595427,COSMO,LICENSED COSMETOLOGIST,11295645.0,ACTIVE,N,NaN,BECKY SUE,L,BURROUGHS,...,NaN,NaN,NaN,N,11/12/2021,NaN,NaN,NaN,NaN,NaN
4,653668,COSMO,LICENSED NAIL TECHNICIAN,169006247,NOT RENEWED,N,NaN,BILL G,L,LETNER,...,NaN,NaN,NaN,N,05/30/2006,NaN,NaN,NaN,NaN,NaN


In [5]:
df.columns

Index(['_id', 'License Type', 'Description', 'License Number',
       'License Status', 'Business', 'Title', 'First Name', 'Middle',
       'Last Name', 'Prefix', 'Suffix', 'Business Name', 'BusinessDBA',
       'Original Issue Date', 'Effective Date', 'Expiration Date', 'City',
       'State', 'Zip', 'County', 'Specialty/Qualifier',
       'Controlled Substance Schedule',
       'Delegated Controlled Substance Schedule', 'Ever Disciplined',
       'LastModifiedDate', 'Case Number', 'Action', 'Discipline Start Date',
       'Discipline End Date', 'Discipline Reason'],
      dtype='object')

## Chart 1

The following chart showcases the total number of licenses for a selected license type with their license status within a selected state. The default state for the chart is Illinois, however, both the license type and 

The following chart showcases the aggregate number of a License Type within each state straited based on the status of the license.

In [6]:
licenses = df[["State", "License Type", "License Status"]]
licenses


,State,License Type,License Status
0,IL,DETECTIVE BOARD,NOT RENEWED
1,IL,DETECTIVE BOARD,NOT RENEWED
2,IL,COSMO,NOT RENEWED
3,IL,COSMO,ACTIVE
4,IL,COSMO,NOT RENEWED
...,...,...,...
9995,IL,DETECTIVE BOARD,NOT RENEWED
9996,IL,DETECTIVE BOARD,TERMINATED CARD RETURNED
9997,IL,COSMO,NOT RENEWED
9998,IL,COSMO,ACTIVE


In [7]:
## drop down 
states = sorted(licenses["State"].unique().tolist())
types = ["All"] + sorted(licenses["License Type"].unique().tolist())

state = alt.param(
    name="selectedState",
    value="IL",
    bind=alt.binding_select(options=states, name="State: ")
)

license_type = alt.param(
    name="selectedType",
    value="All",
    bind=alt.binding_select(options=types, name="License Type: ")
)

In [8]:
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [9]:
fig_1 = alt.Chart(licenses).mark_bar().encode(
    x=alt.X("License Status:N", title="License Status"),
    y=alt.Y("count():Q", title="Number of Licenses",
    axis=alt.Axis(tickMinStep=1)),
    color=alt.value("steelblue"),
    tooltip=["License Status:N", "count():Q"]
).add_params(
    state, license_type
).transform_filter(
    "datum.State == selectedState && (selectedType == 'All' || datum['License Type'] == selectedType)"
).properties(
    title="License Counts by Status", 
    width=550,
    height=350
)
fig_1

alt.Chart(...)

## Chart 1 Overview

The above visualization showcases the number of licenses there are by status for a selected license type in a selected state. The default state presented is Illinois and the default for license type is All. The X-axis uses a nominal encoding to showcase the license status, and the Y-axis uses quantitative to showcase the number of licenses.
I was initally encoding the nominal licenses status as the color with the default color pallete for visualization, however, I realized that the color range could mislead viewers into associating a given color to a particular status when the status the color represented changed based on each independent visualization as the color shifted based on what license statuses were actually available to be represented.

The interactive elements of this visualization as mentioned are the drop downs that inable users to select which state they would like to view and what license type they would like to view information for witin that state. 

Overall this visualization helps to allow users to understand the distribution of professional licneses given their license and state of choice. 

## Chart 2

In [10]:
il = df[df["State"] == "IL"]

top_types = il["License Type"].value_counts().head(10).index
il = il[il["License Type"].isin(top_types)]

il["Active"] = il["License Status"] == "ACTIVE"

percent_active = il.groupby("License Type")["Active"].mean().reset_index()

fig_2 = alt.Chart(percent_active).mark_bar().encode(
    x=alt.X("Active:Q", title="Percentage Active", axis=alt.Axis(format="%")),
    y=alt.Y("License Type:N", sort="-x"),
    tooltip=["License Type:N", alt.Tooltip("Active:Q", format=".1%")]
).properties(
    title="Active Licenses in Illinois"
)

fig_2

alt.Chart(...)

## Chart 2 Overview

The visualization above showcases the percentage of active licenses for each of the ten most common license types within the state of Illinois in the dataset. The license types are ordered in descending order from top to bottom, with the license type having the greatest proportion of active licenses at the top and the license type with the lowest proportion at the bottom. This allows users to quickly understand which license types have a greater proportion of active licenses compared to others. These differences may provide insight into the licensing status and activity of different occupations/businesses within Illinois.



In [11]:
fig_1.save("chart1.html")
fig_2.save("chart2.html")